# Meridian â€” is Paid Social really underperforming?

The attribution dashboard runs on **last touch**. Paid Social gets credit for under 3% of
signups, and the plan on the table is to cut its budget 60% next quarter.

This notebook checks that number. It is not a search for a reason to keep the spend â€” it
reproduces the dashboard figure first, then asks whether last touch is the right model for
the question being asked of it.

**Ground rules from the brief**
- All timestamps UTC. Conversions run 1 Apr â€“ 30 Jun 2026.
- 30-day attribution window: a touch earns credit only if it happened within 30 days
  before the conversion, at or before it.
- A customer who resubscribed is counted under their **first** conversion only.
- Customers who never paid stay in the touch log, but earn no conversion credit.

All logic lives in [`attribution.py`](attribution.py) so this notebook and the dashboard
compute identically.

In [1]:
import pandas as pd

import attribution as attr

pd.set_option("display.width", 160)

touches, conversions = attr.load()
first_convs = attr.first_conversions(conversions)
windowed = attr.windowed_touches(touches, first_convs)

print(f"touch log        {len(touches):>7,} touches   {touches.customer_id.nunique():>6,} customers")
print(f"conversions      {len(conversions):>7,} rows      {conversions.customer_id.nunique():>6,} customers")
print(f"first convs only {len(first_convs):>7,} rows")
print(f"in 30d window    {len(windowed):>7,} touches   {windowed.customer_id.nunique():>6,} customers")

touch log         89,102 touches   40,000 customers
conversions        6,422 rows       5,939 customers
first convs only   5,939 rows
in 30d window     20,533 touches    5,939 customers


## 1. Data quality audit

Before trusting any channel number, check what the two files actually contain. Four things
matter here; the rest of the data is clean.

In [2]:
# a) Repeat subscribers -- the brief says count each customer under their first conversion.
repeat = conversions.customer_id.value_counts()
print(f"customers with >1 subscription start : {(repeat > 1).sum():,}")
print(f"conversion rows                      : {len(conversions):,}")
print(f"conversions after first-only rule    : {len(first_convs):,}  <- denominator for every share below")

# b) Non-converters. They belong in the data, but not in the conversion credit.
never_paid = touches.customer_id.nunique() - first_convs.customer_id.nunique()
print(f"\ncustomers in touch log who never paid: {never_paid:,}")

# c) Dirty columns.
raw_touches = pd.read_csv("data/touches.csv")
print(f"\ndevice values before cleaning        : {sorted(raw_touches.device.unique())}")
print(f"device values after cleaning         : {sorted(touches.device.unique())}")
print(f"campaign nulls (kept as '(none)')    : {raw_touches.campaign.isna().sum():,}")

# d) Things that could have been wrong and are not.
print(f"\nduplicate touch rows                 : {touches.duplicated().sum()}")
print(f"unparseable timestamps               : {touches.touch_ts.isna().sum() + conversions.converted_at.isna().sum()}")
print(f"channel spelling variants            : {len(touches.channel.unique())} distinct, all lowercase snake_case")

customers with >1 subscription start : 483
conversion rows                      : 6,422
conversions after first-only rule    : 5,939  <- denominator for every share below

customers in touch log who never paid: 34,061

device values before cleaning        : ['Mobile', 'Web', 'mobile', 'web']
device values after cleaning         : ['mobile', 'web']
campaign nulls (kept as '(none)')    : 14,831

duplicate touch rows                 : 0
unparseable timestamps               : 0
channel spelling variants            : 9 distinct, all lowercase snake_case


In [3]:
# Every converter keeps at least one eligible touch, so nothing falls into an
# "unattributed" bucket that could hide the answer.
attributable = windowed.customer_id.nunique()
print(f"converters with >=1 eligible touch : {attributable:,} of {len(first_convs):,}")
print(f"converters with no eligible touch  : {len(first_convs) - attributable:,}")

assert set(windowed.customer_id) == set(first_convs.customer_id)
assert (windowed.touch_ts <= windowed.converted_at).all()
assert (windowed.days_before.between(0, 30)).all()
print("\nwindow assertions passed: no touch used is after its conversion or older than 30 days")

converters with >=1 eligible touch : 5,939 of 5,939
converters with no eligible touch  : 0

window assertions passed: no touch used is after its conversion or older than 30 days


## 2. Reproduce the dashboard

If the analysis cannot reproduce the number people are about to act on, nothing else it says
is trustworthy. Last touch gives 100% of the credit to the final eligible touch before the
conversion.

In [4]:
last_touch = attr.share(windowed, "last")
print("LAST-TOUCH SHARE OF SIGNUPS (%)\n")
print(last_touch.round(2).to_string())
print(f"\nPaid Social: {last_touch['paid_social']:.2f}%  (rank {list(last_touch.index).index('paid_social') + 1} of 9)")
print("The dashboard's 'under 3%' reproduces exactly. The arithmetic is correct.")

LAST-TOUCH SHARE OF SIGNUPS (%)

channel
brand_search      34.60
retargeting       23.96
direct            12.11
email             11.40
organic_search     7.81
referral           4.45
paid_social        2.91
youtube            1.78
display            0.98

Paid Social: 2.91%  (rank 7 of 9)
The dashboard's 'under 3%' reproduces exactly. The arithmetic is correct.


In [5]:
# The brief says "under 3% of PAID signups", so check that cut too -- it does not rescue the channel.
paid_only = windowed[windowed.channel.isin(attr.PAID_CHANNELS)]
closers = paid_only[paid_only.position == paid_only.journey_len - 1].channel.value_counts()
paid_share = (closers / closers.sum() * 100).round(2)
print("LAST TOUCH, PAID CHANNELS ONLY (%)\n")
print(paid_share.to_string())

LAST TOUCH, PAID CHANNELS ONLY (%)

channel
brand_search    53.87
retargeting     37.30
paid_social      4.53
youtube          2.78
display          1.52


## 3. The same data under three other models

Nothing changes but the credit rule. Same 5,939 conversions, same 20,533 eligible touches.

- **First touch** â€” 100% to the touch that opened the journey
- **Linear** â€” credit split evenly across every touch
- **Position-based 40/20/40** â€” 40% opener, 40% closer, 20% shared by the middle

In [6]:
comparison = attr.model_comparison(windowed)
comparison.columns = [attr.MODELS[m] for m in comparison.columns]
print("CHANNEL SHARE OF SIGNUPS BY ATTRIBUTION MODEL (%)\n")
print(comparison.to_string())

CHANNEL SHARE OF SIGNUPS BY ATTRIBUTION MODEL (%)



                Last touch (current dashboard)  First touch  Linear  Position-based 40/20/40
channel                                                                                     
paid_social                               2.91        27.29   11.53                    13.59
youtube                                   1.78        17.07    9.08                     9.27
organic_search                            7.81        15.54   14.30                    12.81
referral                                  4.45        12.24   10.74                     9.33
display                                   0.98         8.55    6.00                     5.30
email                                    11.40         7.93   14.07                    11.55
brand_search                             34.60         6.28   15.84                    18.49
retargeting                              23.96         3.05   13.77                    13.60
direct                                   12.11         2.04    4.68   

In [7]:
ranks = {m: list(attr.share(windowed, m).index).index("paid_social") + 1 for m in attr.MODELS}
shares = {m: attr.share(windowed, m)["paid_social"] for m in attr.MODELS}

print("PAID SOCIAL, THE SAME QUARTER, FOUR MODELS\n")
for m, label in attr.MODELS.items():
    print(f"  {label:<28} {shares[m]:>6.2f}%   rank {ranks[m]} of 9")

print(f"\nSpread: {min(shares.values()):.2f}% to {max(shares.values()):.2f}% "
      f"-- a {max(shares.values()) / min(shares.values()):.1f}x range from the model choice alone.")

PAID SOCIAL, THE SAME QUARTER, FOUR MODELS

  Last touch (current dashboard)   2.91%   rank 7 of 9
  First touch                   27.29%   rank 1 of 9
  Linear                        11.53%   rank 5 of 9
  Position-based 40/20/40       13.59%   rank 3 of 9

Spread: 2.91% to 27.29% -- a 9.4x range from the model choice alone.


## 4. Why last touch does this to Paid Social

Three independent pieces of evidence, all pointing the same way: Paid Social acts early, and
last touch only pays the channel that acts last.

In [8]:
# a) Opener vs closer. Paid Social starts journeys it does not finish.
opens = windowed[windowed.position == 0].channel.value_counts()
closes = windowed[windowed.position == windowed.journey_len - 1].channel.value_counts()
touched = windowed.groupby("channel").customer_id.nunique()

role = pd.DataFrame({"converters_touched": touched, "opened": opens, "closed": closes}).fillna(0).astype(int)
role["open_to_close"] = (role.opened / role.closed).round(1)
print("CHANNEL ROLE IN CONVERTING JOURNEYS\n")
print(role.sort_values("open_to_close", ascending=False).to_string())

CHANNEL ROLE IN CONVERTING JOURNEYS

                converters_touched  opened  closed  open_to_close
channel                                                          
youtube                       1678    1014     106            9.6
paid_social                   2063    1621     173            9.4
display                       1177     508      58            8.8
referral                      1940     727     264            2.8
organic_search                2465     923     464            2.0
email                         2447     471     677            0.7
brand_search                  2684     373    2055            0.2
direct                         828     121     719            0.2
retargeting                   2407     181    1423            0.1


In [9]:
# b) Timing. Paid Social lands furthest from the conversion of any channel.
timing = windowed.groupby("channel").days_before.agg(["count", "mean", "median"]).round(2)
timing.columns = ["touches", "mean_days_before", "median_days_before"]
print("HOW LONG BEFORE THE CONVERSION EACH CHANNEL LANDS\n")
print(timing.sort_values("mean_days_before", ascending=False).to_string())

HOW LONG BEFORE THE CONVERSION EACH CHANNEL LANDS

                touches  mean_days_before  median_days_before
channel                                                      
paid_social        2260             14.83               14.46
youtube            1832             14.13               13.50
display            1273             12.85               11.57
referral           2314             11.72               10.20
organic_search     3054             11.26                9.69
email              3067              9.69                7.82
retargeting        2867              6.78                4.29
brand_search       3026              5.87                3.51
direct              840              4.61                2.83


In [10]:
# c) Journey length. Last touch never has a legitimate sole-credit case here.
lengths = windowed.groupby("customer_id").size().value_counts().sort_index()
print("TOUCHES PER CONVERTING JOURNEY\n")
for n, count in lengths.items():
    print(f"  {n} touches : {count:>5,} customers  ({count / lengths.sum() * 100:>5.1f}%)")
print(f"\nSingle-touch journeys: {lengths.get(1, 0)}")
print("Every converting journey has at least two touches, so last touch is always")
print("discarding a known assist -- it is never the whole story for anyone.")

TOUCHES PER CONVERTING JOURNEY

  2 touches : 1,558 customers  ( 26.2%)
  3 touches : 1,528 customers  ( 25.7%)
  4 touches : 1,432 customers  ( 24.1%)
  5 touches : 1,421 customers  ( 23.9%)

Single-touch journeys: 0
Every converting journey has at least two touches, so last touch is always
discarding a known assist -- it is never the whole story for anyone.


In [11]:
# The paths say it outright: Paid Social creates demand that the closers harvest.
paths = attr.journey_paths(windowed)
print("MOST COMMON JOURNEYS THAT INCLUDE PAID SOCIAL\n")
print(paths[paths.str.contains("paid_social")].value_counts().head(10).to_string())

MOST COMMON JOURNEYS THAT INCLUDE PAID SOCIAL

channel
paid_social > brand_search                     135
paid_social > retargeting                       87
paid_social > direct                            48
paid_social > email                             37
paid_social > organic_search > brand_search     35
paid_social > email > brand_search              31
paid_social > email > retargeting               26
paid_social > retargeting > brand_search        25
paid_social > organic_search                    25
paid_social > referral                          24


In [12]:
# The 30-day window penalises Paid Social too, for the same reason: it acts early,
# so more of its touches fall out of the window before the conversion lands.
merged = touches.merge(first_convs[["customer_id", "converted_at"]], on="customer_id")
merged["days_before"] = (merged.converted_at - merged.touch_ts).dt.total_seconds() / 86400

total = merged.groupby("channel").size()
dropped = merged[merged.days_before > 30].groupby("channel").size().reindex(total.index).fillna(0).astype(int)
window_loss = pd.DataFrame({"touches_to_converters": total, "dropped_over_30d": dropped})
window_loss["pct_dropped"] = (window_loss.dropped_over_30d / window_loss.touches_to_converters * 100).round(2)
print("TOUCHES LOST TO THE 30-DAY WINDOW\n")
print(window_loss.sort_values("pct_dropped", ascending=False).to_string())

TOUCHES LOST TO THE 30-DAY WINDOW

                touches_to_converters  dropped_over_30d  pct_dropped
channel                                                             
paid_social                      2545               285        11.20
organic_search                   3087                33         1.07
email                            3100                33         1.06
referral                         2336                22         0.94
display                          1283                10         0.78
retargeting                      2889                22         0.76
youtube                          1844                12         0.65
brand_search                     3031                 5         0.16
direct                            840                 0         0.00


## 5. The counterweight â€” where Paid Social is genuinely weak

An honest review has to test the other direction too. The touch log holds 34,061 customers
who never paid, which lets us compare conversion rates with and without exposure to each
channel.

**This is correlation, not causation**, and it cuts against the closers hardest: people who
already intend to buy go and search the brand, which is why Brand Search and Direct score so
high. But it is the one check that uses the non-converters, and Paid Social does badly on it.

In [13]:
lift = attr.exposure_lift(touches, first_convs)
baseline = first_convs.customer_id.nunique() / touches.customer_id.nunique() * 100
print(f"baseline conversion rate: {baseline:.2f}%\n")
print("CONVERSION RATE BY CHANNEL EXPOSURE\n")
print(lift.to_string(index=False))
print("\nPaid Social has the LOWEST exposure lift of all nine channels. It is a broad-reach")
print("channel -- it touches 37% of all known customers, so seeing it barely discriminates.")

baseline conversion rate: 14.85%

CONVERSION RATE BY CHANNEL EXPOSURE

       channel  customers_exposed  cvr_exposed_pct  cvr_not_exposed_pct  lift_x
        direct               1517            54.58                13.28    4.11
  brand_search               6804            39.49                 9.80    4.03
   retargeting               8120            29.79                11.04    2.70
         email              11374            21.65                12.15    1.78
organic_search              12789            19.42                12.70    1.53
      referral              10001            19.55                13.28    1.47
       display               6374            18.61                14.13    1.32
       youtube               9252            18.24                13.83    1.32
   paid_social              14797            15.62                14.40    1.08

Paid Social has the LOWEST exposure lift of all nine channels. It is a broad-reach
channel -- it touches 37% of all known custom

In [14]:
# But journeys that START with Paid Social do convert above baseline.
converter_ids = set(first_convs.customer_id)
opener = (
    touches.sort_values(["customer_id", "touch_ts"])
    .groupby("customer_id")
    .agg(first_channel=("channel", "first"))
)
opener["converted"] = opener.index.isin(converter_ids)

by_opener = opener.groupby("first_channel").converted.agg(["size", "sum", "mean"])
by_opener.columns = ["customers", "converted", "cvr"]
by_opener["cvr_pct"] = (by_opener.cvr * 100).round(2)
print("CONVERSION RATE BY THE CHANNEL THAT OPENED THE JOURNEY\n")
print(by_opener[["customers", "converted", "cvr_pct"]].sort_values("cvr_pct", ascending=False).to_string())
print(f"\nbaseline: {baseline:.2f}%")

CONVERSION RATE BY THE CHANNEL THAT OPENED THE JOURNEY

                customers  converted  cvr_pct
first_channel                                
direct                557        121    21.72
paid_social          9543       1892    19.83
youtube              5279        957    18.13
brand_search         2493        368    14.76
display              3148        462    14.68
referral             4912        674    13.72
organic_search       6528        863    13.22
email                4796        432     9.01
retargeting          2744        170     6.20

baseline: 14.85%


In [15]:
# Revenue-weighted, in case Paid Social buys cheaper customers. It does not --
# plan mix is flat across every channel.
windowed_rev = windowed.copy()
rev = attr.model_comparison(windowed_rev, value_col="amount_usd")
rev.columns = [attr.MODELS[m] for m in rev.columns]
print("REVENUE-WEIGHTED SHARE (%)\n")
print(rev.to_string())

plan_mix = (
    windowed[windowed.position == 0]
    .groupby("channel").plan.value_counts(normalize=True).unstack().round(3)
)
print("\nPLAN MIX BY OPENING CHANNEL (annual = $99, monthly = $12)\n")
print(plan_mix.to_string())
print("\nFlat at roughly 50/50 everywhere -- no channel is buying materially worse customers.")

REVENUE-WEIGHTED SHARE (%)

                Last touch (current dashboard)  First touch  Linear  Position-based 40/20/40
channel                                                                                     
paid_social                               3.01        27.45   11.71                    13.72
youtube                                   1.62        16.91    8.95                     9.13
organic_search                            7.66        15.38   14.15                    12.67
referral                                  4.73        12.36   10.96                     9.55
display                                   0.89         8.55    5.99                     5.29
email                                    11.08         7.95   13.92                    11.39
brand_search                             34.73         6.10   15.70                    18.41
retargeting                              23.82         3.12   13.81                    13.58
direct                                   1

In [16]:
# Stability check: is this a one-month artifact? No -- it holds all three months.
windowed["month"] = windowed.converted_at.dt.to_period("M")
rows = []
for month, grp in windowed.groupby("month"):
    ls = attr.share(grp, "last")
    fs = attr.share(grp, "first")
    rows.append({
        "month": str(month),
        "conversions": grp.customer_id.nunique(),
        "paid_social_last_pct": round(ls.get("paid_social", 0), 2),
        "paid_social_first_pct": round(fs.get("paid_social", 0), 2),
    })
print("PAID SOCIAL BY MONTH\n")
print(pd.DataFrame(rows).to_string(index=False))
print("\nStable across April, May and June. This is structural, not a blip.")

PAID SOCIAL BY MONTH

  month  conversions  paid_social_last_pct  paid_social_first_pct
2026-04         1879                  2.87                  27.78
2026-05         2089                  3.06                  25.75
2026-06         1971                  2.79                  28.46

Stable across April, May and June. This is structural, not a blip.


---
## The five answers

In [17]:
last_s = attr.share(windowed, "last")
first_s = attr.share(windowed, "first")
linear_s = attr.share(windowed, "linear")
position_s = attr.share(windowed, "position")

ps_touched = windowed[windowed.channel == "paid_social"].customer_id.nunique()
ps_opened = int((windowed[windowed.position == 0].channel == "paid_social").sum())
ps_closed = int((windowed[windowed.position == windowed.journey_len - 1].channel == "paid_social").sum())
n_conv = len(first_convs)
ps_lift = attr.exposure_lift(touches, first_convs).set_index("channel").loc["paid_social", "lift_x"]

print("=" * 78)
print("MERIDIAN â€” PAID SOCIAL ATTRIBUTION REVIEW â€” FIVE ANSWERS".center(78))
print("=" * 78)

print(f'''
1. IS THE DASHBOARD'S NUMBER CORRECT?
   Yes, and it is still misleading. Paid Social earns {last_s['paid_social']:.2f}% of signups under
   last touch, rank {list(last_s.index).index('paid_social') + 1} of 9. Reproduced exactly from {n_conv:,} first conversions and
   {len(windowed):,} touches inside the 30-day window.
   Method: attribution.share(windowed, "last") â€” 100% of credit to the final
   eligible touch per customer.

2. WHAT DOES PAID SOCIAL LOOK LIKE ON FIRST TOUCH?
   {first_s['paid_social']:.2f}% of signups â€” rank {list(first_s.index).index('paid_social') + 1} of 9, the single biggest opener of
   converting journeys. Brand Search, which last touch ranks first at
   {last_s['brand_search']:.2f}%, falls to {first_s['brand_search']:.2f}% here.
   Method: attribution.share(windowed, "first").

3. AND UNDER MODELS THAT CREDIT THE WHOLE JOURNEY?
   Linear {linear_s['paid_social']:.2f}% (rank {list(linear_s.index).index('paid_social') + 1}), position-based 40/20/40 {position_s['paid_social']:.2f}% (rank {list(position_s.index).index('paid_social') + 1}).
   Across all four models Paid Social's contribution spans {last_s['paid_social']:.2f}%â€“{first_s['paid_social']:.2f}%,
   a {first_s['paid_social'] / last_s['paid_social']:.1f}x range driven entirely by the choice of model.
   Method: attribution.model_comparison(windowed).

4. WHAT IS PAID SOCIAL ACTUALLY DOING?
   It touched {ps_touched:,} of {n_conv:,} converters ({ps_touched / n_conv * 100:.1f}%). It OPENED {ps_opened:,}
   converting journeys and CLOSED only {ps_closed:,} â€” {ps_opened / ps_closed:.1f} opens for every close.
   It lands {windowed[windowed.channel == 'paid_social'].days_before.mean():.1f} days before conversion on average, the earliest of any
   channel; Brand Search lands at {windowed[windowed.channel == 'brand_search'].days_before.mean():.1f} days. Commonest paths are
   paid_social > brand_search and paid_social > retargeting.
   Last touch pays the harvester, never the channel that planted.
   Method: position == 0 vs position == journey_len - 1 on the windowed touches.

5. SHOULD THE BUDGET BE CUT 60%?
   No â€” not on this evidence. The 60% cut rests on the one model that is wrong
   by construction: every converting journey here has 2+ touches, so last touch
   is always discarding a known assist, and it discards Paid Social's most.
   Cutting 60% would remove the top opener of converting journeys to protect
   channels that mostly harvest demand Paid Social created.
   But do not read this as "spend more". Paid Social's exposure lift is {ps_lift:.2f}x,
   the LOWEST of nine channels, and this export contains NO SPEND DATA, so no
   CAC or ROAS can be computed and efficiency is unproven either way.
   Recommendation: hold the budget one quarter, run a geo holdout to get a
   causal read, and re-cut the dashboard to position-based with last touch
   kept as a secondary column.
''')
print("=" * 78)

         MERIDIAN â€” PAID SOCIAL ATTRIBUTION REVIEW â€” FIVE ANSWERS         

1. IS THE DASHBOARD'S NUMBER CORRECT?
   Yes, and it is still misleading. Paid Social earns 2.91% of signups under
   last touch, rank 7 of 9. Reproduced exactly from 5,939 first conversions and
   20,533 touches inside the 30-day window.
   Method: attribution.share(windowed, "last") â€” 100% of credit to the final
   eligible touch per customer.

2. WHAT DOES PAID SOCIAL LOOK LIKE ON FIRST TOUCH?
   27.29% of signups â€” rank 1 of 9, the single biggest opener of
   converting journeys. Brand Search, which last touch ranks first at
   34.60%, falls to 6.28% here.
   Method: attribution.share(windowed, "first").

3. AND UNDER MODELS THAT CREDIT THE WHOLE JOURNEY?
   Linear 11.53% (rank 5), position-based 40/20/40 13.59% (rank 3).
   Across all four models Paid Social's contribution spans 2.91%â€“27.29%,
   a 9.4x range driven entirely by the choice of model.
   Method: attribution.model_comparison(windowed)